# Inspección de enlaces de datos abiertos de MiBici

Esta libreta explora la página de [datos abiertos de MiBici](https://mibici.net/en/open-data/) (sistema de bicicleta pública de Guadalajara) para identificar qué archivos CSV publica y cómo están organizados.

Pasos:
1. Descargar el HTML de la página.
2. Extraer todos los enlaces que apuntan a archivos `.csv`.
3. Revisar una muestra de los enlaces encontrados.
4. Ordenar los archivos por nombre y clasificarlos (viajes, nomenclatura, otros).
5. Verificar que no haya nombres de archivo duplicados.

## 1. Importaciones

In [2]:
# urljoin convierte rutas relativas (p. ej. "/files/data.csv") en URLs absolutas
from urllib.parse import urljoin

# requests: cliente HTTP para descargar la página
import requests
# BeautifulSoup: parser de HTML para extraer los enlaces
from bs4 import BeautifulSoup

## 2. Descarga de la página

Se hace una petición HTTP GET a la página de datos abiertos. Se envían encabezados que imitan a un navegador real (`User-Agent`, `Accept`, `Accept-Language`), ya que algunos sitios bloquean o responden distinto a clientes automatizados.

Se imprimen el código de estado, la URL final (por si hubo redirecciones) y el tipo de contenido, para confirmar que se recibió HTML.

In [4]:
# Página de datos abiertos de MiBici (versión en inglés)
OPEN_DATA_URL = "https://mibici.net/en/open-data/"

# Encabezados que simulan un navegador para evitar bloqueos a clientes automatizados
headers = {
    # Identifica la petición como Chrome en macOS
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/141.0 Safari/537.36"
    ),
    # Tipos de contenido aceptados, con preferencia por HTML
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    # Idiomas preferidos: español de México, español, inglés
    "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
}

# Descarga el HTML; timeout evita que la petición se quede colgada indefinidamente
response = requests.get(
    OPEN_DATA_URL,
    headers=headers,
    timeout=30,
)

# Código HTTP (200 = OK), URL final tras redirecciones y tipo de contenido recibido
print("Status:", response.status_code)
print("URL:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))

Status: 200
URL: https://mibici.net/en/open-data/
Content-Type: text/html; charset=utf-8


## 3. Extracción de enlaces a CSV

Se parsea el HTML con BeautifulSoup y se buscan todas las etiquetas `<a>` con atributo `href`. Se conservan solo las que terminan en `.csv` (ignorando mayúsculas y parámetros de consulta), se convierten en URLs absolutas y se eliminan duplicados.

In [5]:
# Parsea el HTML descargado
soup = BeautifulSoup(response.text, "html.parser")

# Recolecta las URLs de todos los enlaces que apuntan a archivos CSV:
# - find_all("a", href=True): solo etiquetas <a> que tienen atributo href
# - split("?")[0]: descarta parámetros de consulta antes de revisar la extensión
# - urljoin: convierte enlaces relativos en absolutos respecto a OPEN_DATA_URL
# - el set {...} elimina duplicados y sorted() deja la lista ordenada
csv_urls = sorted(
    {
        urljoin(OPEN_DATA_URL, link["href"])
        for link in soup.find_all("a", href=True)
        if link["href"].lower().split("?")[0].endswith(".csv")
    }
)

# Total de URLs de CSV únicas encontradas
print(f"CSV encontrados: {len(csv_urls)}")

CSV encontrados: 143


## 4. Revisión de los enlaces

Se muestra una muestra del inicio y del final de la lista. Como `csv_urls` está ordenada por la URL completa, el orden depende del identificador numérico de la carpeta (`/files/<id>/`) y no de la fecha del archivo; por eso los periodos aparecen mezclados. Esto se corrige en la siguiente sección.

### Primeros 10 enlaces

In [6]:
# Muestra las primeras 10 URLs de la lista ordenada
for url in csv_urls[:10]:
    print(url)

https://mibici.net/site/assets/files/10088/datos_abiertos_2021_10.csv
https://mibici.net/site/assets/files/101986/datos_abiertos_2025_03.csv
https://mibici.net/site/assets/files/1020/datos_abiertos_2016_01-1.csv
https://mibici.net/site/assets/files/1021/datos_abiertos_2016_02.csv
https://mibici.net/site/assets/files/1022/datos_abiertos_2017_01.csv
https://mibici.net/site/assets/files/1023/datos_abiertos_2016_03.csv
https://mibici.net/site/assets/files/1029/datos_abiertos_2016_04.csv
https://mibici.net/site/assets/files/103813/datos_abiertos_2025_04.csv
https://mibici.net/site/assets/files/105513/datos_abiertos_2025_05.csv
https://mibici.net/site/assets/files/1058/datos_abiertos_2016_05.csv


### Últimos 10 enlaces

In [7]:
# Muestra las últimas 10 URLs de la lista ordenada
for url in csv_urls[-10:]:
    print(url)

https://mibici.net/site/assets/files/83901/datos_abiertos_2024_06.csv
https://mibici.net/site/assets/files/8461/datos_abiertos_2021_09.csv
https://mibici.net/site/assets/files/85500/datos_abiertos_2024_07.csv
https://mibici.net/site/assets/files/87665/datos_abiertos_2024_08.csv
https://mibici.net/site/assets/files/89736/datos_abiertos_2024_09.csv
https://mibici.net/site/assets/files/91758/datos_abiertos_2024_10.csv
https://mibici.net/site/assets/files/93516/datos_abiertos_2024_11.csv
https://mibici.net/site/assets/files/94962/datos_abiertos_2024_12.csv
https://mibici.net/site/assets/files/97653/datos_abiertos_2025_01.csv
https://mibici.net/site/assets/files/99879/datos_abiertos_2025_02.csv


## 5. Ordenar por nombre de archivo

Se extrae el nombre de archivo de cada URL (la última parte de la ruta) y se ordena la lista por ese nombre. Como los archivos siguen el patrón `datos_abiertos_AAAA_MM.csv`, el orden alfabético coincide con el orden cronológico.

In [8]:
from pathlib import Path
from urllib.parse import urlparse

# Lista de diccionarios con el nombre de archivo y su URL de descarga.
# urlparse(url).path obtiene solo la ruta; Path(...).name toma su último segmento.
csv_files = [
    {
        "filename": Path(urlparse(url).path).name,
        "url": url,
    }
    for url in csv_urls
]

# Ordena por nombre de archivo (equivale a orden cronológico por el patrón AAAA_MM)
csv_files = sorted(csv_files, key=lambda item: item["filename"])

print(f"Total CSV: {len(csv_files)}")

# Muestra los primeros 20 nombres para verificar el orden
for item in csv_files[:20]:
    print(item["filename"])

Total CSV: 143
datos_abiertos_2014_12.csv
datos_abiertos_2015_01.csv
datos_abiertos_2015_02.csv
datos_abiertos_2015_03.csv
datos_abiertos_2015_04.csv
datos_abiertos_2015_05.csv
datos_abiertos_2015_06.csv
datos_abiertos_2015_07.csv
datos_abiertos_2015_08.csv
datos_abiertos_2015_09.csv
datos_abiertos_2015_10.csv
datos_abiertos_2015_11.csv
datos_abiertos_2015_12.csv
datos_abiertos_2016_01-1.csv
datos_abiertos_2016_02.csv
datos_abiertos_2016_03.csv
datos_abiertos_2016_04.csv
datos_abiertos_2016_05.csv
datos_abiertos_2016_06.csv
datos_abiertos_2016_07.csv


## 6. Clasificación de los archivos

Los archivos se agrupan según su prefijo:
- `datos_abiertos_*`: registros de viajes (uno por mes).
- `nomenclatura_*`: catálogo de estaciones.
- Cualquier otro archivo queda en `other_files` para revisarlo manualmente.

In [9]:
# Archivos de viajes: prefijo "datos_abiertos_"
trip_files = [
    item for item in csv_files
    if item["filename"].startswith("datos_abiertos_")
]

# Archivos de nomenclatura (catálogo de estaciones): prefijo "nomenclatura_"
nomenclature_files = [
    item for item in csv_files
    if item["filename"].startswith("nomenclatura_")
]

# Todo lo que no encaja en las categorías anteriores
other_files = [
    item for item in csv_files
    if item not in trip_files and item not in nomenclature_files
]

# Conteo por categoría
print("Viajes:", len(trip_files))
print("Nomenclatura:", len(nomenclature_files))
print("Otros:", len(other_files))

Viajes: 142
Nomenclatura: 1
Otros: 0


### Archivos sin clasificar

Lista los archivos que no son de viajes ni de nomenclatura (si los hay), para decidir qué hacer con ellos.

In [10]:
# Imprime nombre y URL de cada archivo no clasificado
for item in other_files:
    print(item["filename"], "->", item["url"])

## 7. Verificación de nombres duplicados

Distintas URLs podrían terminar en el mismo nombre de archivo (por ejemplo, el mismo mes publicado en dos carpetas). Si eso pasa, al descargarlos uno sobrescribiría al otro, así que se comprueba que cada nombre sea único.

In [11]:
from collections import Counter

# Cuenta cuántas veces aparece cada nombre de archivo
filename_counts = Counter(item["filename"] for item in csv_files)

# Conserva solo los nombres que aparecen más de una vez
duplicates = {
    filename: count
    for filename, count in filename_counts.items()
    if count > 1
}

print("Nombres duplicados:", len(duplicates))

# Detalle de cada nombre duplicado y cuántas veces aparece
for filename, count in duplicates.items():
    print(filename, count)

Nombres duplicados: 0


## 8. Archivo de nomenclatura

Muestra el nombre y la URL del catálogo de estaciones.

In [12]:
# Imprime nombre y URL de cada archivo de nomenclatura
for item in nomenclature_files:
    print(item["filename"])
    print(item["url"])

nomenclatura_2026_09.csv
https://mibici.net/site/assets/files/1118/nomenclatura_2026_09.csv
